In [0]:
# Channels info
df_channels = spark.table("workspace_youtube_1.silver.channels")

# Videos info
df_videos = spark.table("workspace_youtube_1.silver.videos")

In [0]:
from pyspark.sql.functions import sum, max, count, col

df_gold = df_videos.groupBy("channel_id", "channel_title") \
    .agg(
        count("video_id").alias("total_videos"),
        max("published_at").alias("latest_video_date")
    )

In [0]:
df_gold = df_gold.join(
    df_channels.select("channel_id", "subscribers"),
    on="channel_id",
    how="left"
)

In [0]:
from pyspark.sql.functions import current_timestamp

df_gold = df_gold.withColumn("ingestion_time", current_timestamp())
df_gold.display()

In [0]:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace_youtube_1.gold.channel_analytics")

In [0]:
spark.sql("SELECT * FROM workspace_youtube_1.gold.channel_analytics").show(truncate=False)